In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v0_6 import BioJepa, BioJepaConfig
from dataloader_v0_6 import PretrainLoader, AlignmentLoader, TrainingLoader
from training_v0_6 import create_model, load_feature_banks, run_pretraining, run_alignment, run_full_training, train_linear_decoder, maybe_compile
from config_v0_6 import PretrainConfig, AlignmentConfig, FullTrainingConfig, DecoderConfig, DataConfig
from evals.evals import EvalContext, run_pretraining_evals, run_alignment_evals, run_full_model_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('/home/ubuntu/data/v0_6')
ref_root = Path('/home/ubuntu/data/reference_data')

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoint',
    ref_dir = ref_root,
    eval_results_dir=data_root / 'eval_results'
)

using cuda


In [3]:
# Model architecture
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=6,
    heads=4,
    embed_dim=256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=2.38,
    film_linear_multiple=0.81,
    sim_coeff=40.5,
    std_coeff=40.5,
    cov_coeff=1.62,
    pert_latent_dim= 320,
    pert_mode_dim= 64,
)

# Training configs
pt_cfg = PretrainConfig(epochs=50, lr=1e-4, batch_size=64) 
align_cfg = AlignmentConfig(epochs=10000, lr=7.6e-4, batch_size=64, weight_decay=0.011)
full_cfg = FullTrainingConfig(epochs=20, predictor_lr=1e-4, batch_size=32, mask_anneal_pct=0.15) 
decoder_cfg = DecoderConfig(epochs=10, lr=1e-3, batch_size=32) 

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)
seq_banks, target_bank = load_feature_banks(data_cfg, device)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters() if p.requires_grad):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters() if p.requires_grad):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters() if p.requires_grad):,}')

Loaded DNA embeddings: torch.Size([11643, 1536])
Loaded chemical embeddings: torch.Size([188, 1536])
Loaded target embeddings: torch.Size([9975, 320])
Student/Teacher: 7,979,266
ACpredictor: 10,036,224
PerturbationComposer: 1,478,976


### Load Model 

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_6_full_final.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

### Encoder Training Evals

In [6]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': pt_cfg.batch_size, 'seed': SEED
})
pt_eval_results = run_pretraining_evals(eval_ctx)

save_report(pt_eval_results, data_cfg.eval_results_dir / 'pretraining_eval_report.json')
pt_eval_results

Using cuda
found 121 shards for split test


batch_invariance: Extracting embeddings: 100%|███████████████| 4840/4840 [05:59<00:00, 13.45it/s]


Training classifiers...
batch_invariance: Batch=0.0443 (16.7x), Pert=0.0261 (28.4x)
batch_invariance summary: global_ratio=0.588, within_dataset_macro_ratio=0.681
Loading KEGG_2026...
  352 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
gene_embedding_pathways: KEGG ratio=1.1210
essential_gene_prediction: Pearson=0.3138, AUROC=0.7099
found 121 shards for split test


cell_type_probing: Extracting embeddings: 100%|██████████████| 4840/4840 [05:54<00:00, 13.66it/s]


Training cell type classifier...
cell_type_probing: Accuracy=0.8790 (2.6x chance), Macro F1=0.7006
found 121 shards for split test


...

embedding_consistency: Extracting embeddings: 100%|██████████| 4840/4840 [05:55<00:00, 13.61it/s]


embedding_consistency: Computing intra-distances for 1127 perturbations...
embedding_consistency: Computing 5000 inter-distances...
embedding_consistency: Intra=5.6925, Inter=5.2420, Ratio=0.92x
embedding_consistency: Computing for dataset adamson...
embedding_consistency: Computing for dataset k562e_raw...
embedding_consistency: Computing for dataset k562gw...
embedding_consistency: Computing for dataset norman...
embedding_consistency: Computing for dataset sciplex...
found 121 shards for split test


latent_space_health: Extracting embeddings: 100%|████████████| 4840/4840 [05:53<00:00, 13.70it/s]


latent_space_health: Eff_dim_90=4/256, Mean_var=0.1431, Isotropy=0.000000
Saved report to /home/ubuntu/data/v0_6/eval_results/pretraining_eval_report.json


{'batch_invariance': {'config': {'samples': 309760,
   'embedding_dim': 256,
   'num_batches': 376,
   'num_perturbations': 1089},
  'batch_classifier': {'accuracy': 0.044308496900826444,
   'chance': 0.0026595744680851063,
   'above_chance_ratio': 16.659994834710744},
  'perturbation_classifier': {'accuracy': 0.026068569214876033,
   'chance': 0.0009182736455463728,
   'above_chance_ratio': 28.388671875},
  'invariance_ratio': 0.5883424408014573,
  'by_dataset': {'k562e_raw': {'config': {'samples': 49066,
     'embedding_dim': 256,
     'num_batches': 48,
     'num_perturbations': 286},
    'batch_classifier': {'accuracy': 0.0654167515793764,
     'chance': 0.020833333333333332,
     'above_chance_ratio': 3.1400040758100674},
    'perturbation_classifier': {'accuracy': 0.016405135520684736,
     'chance': 0.0034965034965034965,
     'above_chance_ratio': 4.6918687589158345},
    'invariance_ratio': 0.2507788161993769},
   'k562gw': {'config': {'samples': 178474,
     'embedding_dim': 

In [7]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Alignment Training Eval

In [8]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': align_cfg.batch_size, 'seed': SEED
})
align_eval_results = run_alignment_evals(align_eval_ctx)

save_report(align_eval_results, data_cfg.eval_results_dir / 'alignment_eval_report.json')
align_eval_results

Using cuda
Loaded 10797 v0.6 alignment pairs
Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Encoded DNA sequences: torch.Size([11643, 320])
Encoded chemical sequences: torch.Size([188, 320])
Loaded target bank: torch.Size([9975, 320])
Encoded protein targets: torch.Size([9975, 320])
seq_to_target_retrieval: dna_mrr=0.0032
cross_modality_target_consistency: Within=0.6953, Between=0.6252, Ratio=1.11x
seq_target_gap_analysis: dna_gap=0.91
paired_alignment_quality: dna_sim=0.6987
mode_sensitivity: Classification_acc=0.8857 (6.2x chance)
fusion_quality: Fused_var=0.0968, Seq_var=0.0622, Target_var=0.0269
missing_data_robustness: Fused_MRR=0.2865, Seq_only=0.0017, Target_only=1.0000
found 121 shards for split test


multi_pert_alignment: Scanning for multi-pert: 100%|██████████| 500/500 [00:03<00:00, 135.11it/s]


Loaded 27264 HGNC gene family annotations
target_family_probing: seq_only=0.0787, target_only=0.3229, fused=0.2542
action_vector_pathways DNA: ratio=1.001103104216978
Saved report to /home/ubuntu/data/v0_6/eval_results/alignment_eval_report.json


{'seq_to_target_retrieval': {'config': {'n_targets': 9975},
  'by_modality': {'dna': {'mrr': 0.0032327098624400715,
    'median_rank': 3508.0,
    'mean_rank': 3923.341077979494,
    'n_queries': 10631,
    'n_targets': 9975,
    'recall_at_k': {'1': 0.0006584516978647352,
     '5': 0.0033863230175900667,
     '10': 0.00498541999811871,
     '20': 0.009218323770106293,
     '50': 0.02003574452074123}}}},
 'cross_modality_target_consistency': {'config': {'n_valid_targets': 915,
   'n_within_pairs': 1452,
   'n_between_pairs': 5000},
  'metrics': {'within_target_sim': 0.6952723264694214,
   'between_target_sim': 0.6252175957880914,
   'consistency_ratio': 1.112048559018921}},
 'seq_target_gap_analysis': {'target_variance': 8.119363784790039,
  'n_targets': 9975,
  'dna': {'seq_variance': 20.38433837890625,
   'centroid_distance': 2.2549543380737305,
   'mean_within_seq': 6.235803158495194,
   'mean_seq_to_target': 5.688068389892578,
   'gap_ratio': 0.9121629155570085,
   'n_sequences': 1

In [9]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()

### ACPredictor Eval

In [6]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_v0_6_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
decoder.load_state_dict(decoder_sd)

<All keys matched successfully>

In [8]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': full_cfg.batch_size, 'seed': SEED,
})
full_eval_results = run_full_model_evals(eval_ctx)

save_report(full_eval_results, data_cfg.eval_results_dir / 'full_model_eval_report.json')
full_eval_results

Using cuda
found 121 shards for split test


Running test inference:   0%|                                           | 0/9680 [00:00<?, ?it/s]

Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Loaded target bank: torch.Size([9975, 320])


Running test inference:   8%|██▌                              | 745/9680 [03:14<21:30,  6.92it/s]/home/ubuntu/code/biojepa/evals/evals.py:560: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(p_top, t_top)
Running test inference: 100%|████████████████████████████████| 9680/9680 [42:21<00:00,  3.81it/s]


Aggregated 1119 single-pert, 8 multi-pert perturbations, 309760 samples, 121 shards
  adamson: 11 perturbations, 5331 samples
  k562e_raw: 286 perturbations, 49066 samples
  k562gw: 1053 perturbations, 178474 samples
  sciplex: 54 perturbations, 74203 samples
Cached test inference to /home/ubuntu/data/v0_6/test_inference_cache (121 shards)
expression_prediction: Pearson=0.9904, R2=0.9798, Centroid_acc=0.5967
gene_level_analysis: Dir_acc=0.9935, Top50_acc=0.7929


...

perturbation_retrieval (chemical): 100%|█████████████████████████| 54/54 [00:39<00:00,  1.37it/s]


perturbation_retrieval (chemical): MRR=0.0235


perturbation_retrieval (target_only): 100%|████████████████████████| 3/3 [01:55<00:00, 38.66s/it]


perturbation_retrieval (target_only): MRR=0.0014
uncertainty_calibration: ECE=0.1993, Monotonicity=55.56%
Loading KEGG_2026...
  352 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
moa_matching expression: Within=0.5979, Between=0.5517, Gap=0.0462, Ratio=1.0837x
moa_matching latent: Within=0.9712, Between=0.9701, Gap=0.0011, Ratio=1.0011x
Loaded Norman combo mapping: 132 combos
Loaded Norman single-gene deltas: 105 genes
Loaded Norman GI subtypes: 88 combos
combination_perturbation: 8 combo perts, 2686 samples, 8 additive baseline, 6 GI-labeled
dose_response: monotonicity=51.23%, spearman=0.0465
Saved report to /home/ubuntu/data/v0_6/eval_results/full_model_eval_report.json


{'expression_prediction': {'config': {'test_perturbations': 1119,
   'genes': 10000,
   'test_samples': 309760},
  'sample_level': {'mse': 0.17162137919312204,
   'pearson_r_top20': 0.8980039304965769},
  'perturbation_level': {'r2_all_genes': {'mean': 0.9798075005984711,
    'median': 0.9850167632102966},
   'r2_top50_degs': {'mean': 0.8599820964244778, 'median': 0.8943163752555847},
   'mse': {'mean': 0.0036781560629606247, 'median': 0.00318646221421659},
   'pearson_all_genes': {'mean': 0.9903552410216924,
    'median': 0.9927735328674316},
   'pearson_delta_all_genes': {'mean': 0.3511938888736076,
    'median': 0.34927505254745483},
   'pearson_top50_degs': {'mean': 0.5360258915499052,
    'median': 0.5626335144042969}},
  'centroid_accuracy': {'accuracy': 0.5967117988394585, 'n_groups': 1034},
  'vs_baseline': {'beat_rate': 0.05987488829311886, 'n_evaluated': 1119},
  'severity': {'pearson_r': 0.7931576371192932,
   'spearman_r': 0.6291477795538212},
  'error_by_magnitude': {'0-0.

In [12]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()